In [198]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor
import pickle
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

In [95]:
df = pd.read_csv(r"C:\Users\SART\Desktop\Practise\csvs\archive\train_regression.csv")
pd.set_option('display.max_columns', None)
df.head()

,id,host_id,host_since,host_location,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_neighbourhood,host_listings_count,host_total_listings_count,host_verifications,host_has_profile_pic,host_identity_verified,neighbourhood_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms_text,beds,price,minimum_nights,maximum_nights,minimum_minimum_nights,maximum_minimum_nights,minimum_maximum_nights,maximum_maximum_nights,minimum_nights_avg_ntm,maximum_nights_avg_ntm,has_availability,availability_30,availability_60,availability_90,availability_365,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,first_review,last_review,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,5.387292e+07,366516391,4/9/2018,"Chicago, IL",within an hour,96%,95%,f,Back of the Yards,11,13,"['email', 'phone']",t,t,New City,41.794240,-87.661380,Private room in rental unit,Private room,1,1 shared bath,1.0,$30.00,32,1125,32,32,1125,1125,32.0,1125.0,t,30,60,90,365,18,0,0,6/16/2019,8/12/2022,4.94,5.00,4.89,5.00,5.00,4.28,4.83,f,9,1,8,0,0.33
1,3.926966e+07,50276775,12/23/2014,NaN,within an hour,100%,97%,f,Clearwater Beach,1141,2346,"['email', 'phone']",t,t,Near North Side,41.892749,-87.620711,Room in hotel,Private room,12,3 baths,3.0,"$1,247.00",32,365,32,32,365,365,32.0,365.0,t,25,49,66,135,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,t,58,0,58,0,NaN
2,5.564810e+17,681680651,3/8/2020,NaN,within an hour,100%,100%,f,Lake View East,1,1,['phone'],t,t,Lake View,41.948160,-87.652309,Entire rental unit,Entire home/apt,6,1 bath,3.0,$214.00,2,45,2,2,45,45,2.0,45.0,t,0,0,0,0,14,14,1,7/21/2023,12/3/2023,4.87,5.00,5.00,5.00,5.00,4.93,4.80,f,1,1,0,0,2.78
3,5.239728e+07,94345291,10/22/2015,"Chicago, IL",within an hour,100%,98%,f,Bucktown,55,58,"['email', 'phone', 'work_email']",t,t,Hyde Park,41.801030,-87.597160,Entire rental unit,Entire home/apt,2,1 bath,1.0,$59.00,2,180,2,2,180,180,2.0,180.0,t,23,53,83,355,13,13,3,9/2/2023,12/6/2023,4.08,4.08,4.00,4.38,4.23,4.54,3.85,t,55,55,0,0,3.61
4,7.459490e+17,790998106,4/4/2021,"Chicago, IL",within an hour,100%,100%,t,Near South Side,74,75,"['email', 'phone']",t,t,Near South Side,41.855770,-87.624130,Entire rental unit,Entire home/apt,6,2 baths,3.0,$151.00,2,365,2,2,1125,1125,2.0,1125.0,t,0,0,0,0,64,42,3,5/6/2022,12/6/2023,4.80,4.74,4.75,4.83,4.85,4.74,4.72,t,74,74,0,0,3.24


In [96]:
df['host_location']

0        Chicago, IL
1                NaN
2                NaN
3        Chicago, IL
4        Chicago, IL
            ...     
4995     Chicago, IL
4996     Chicago, IL
4997     Chicago, IL
4998    New York, NY
4999     Chicago, IL
Name: host_location, Length: 5000, dtype: str

In [97]:
df['host_location'] = df['host_location'].str.split(',').str[0].str.strip()

In [98]:
df['host_location']

0        Chicago
1            NaN
2            NaN
3        Chicago
4        Chicago
          ...   
4995     Chicago
4996     Chicago
4997     Chicago
4998    New York
4999     Chicago
Name: host_location, Length: 5000, dtype: object

In [99]:
df['host_response_rate'] = df['host_response_rate'].str.replace('%', '').str.strip()

In [100]:
df['host_acceptance_rate'] = df['host_acceptance_rate'].str.replace('%', '').str.strip()

In [101]:
a = df.isnull().sum().sort_values(ascending=False)
b = a[a > 0]
b

review_scores_communication    1042
first_review                   1042
reviews_per_month              1042
last_review                    1042
review_scores_cleanliness      1042
review_scores_rating           1041
review_scores_checkin          1041
review_scores_accuracy         1041
review_scores_value            1041
review_scores_location         1041
host_location                   958
host_response_time              418
host_response_rate              418
host_acceptance_rate            291
host_neighbourhood              127
beds                             48
host_is_superhost                23
bathrooms_text                    4
dtype: int64

In [102]:
df[b.index]

,review_scores_communication,first_review,reviews_per_month,last_review,review_scores_cleanliness,review_scores_rating,review_scores_checkin,review_scores_accuracy,review_scores_value,review_scores_location,host_location,host_response_time,host_response_rate,host_acceptance_rate,host_neighbourhood,beds,host_is_superhost,bathrooms_text
0,5.00,6/16/2019,0.33,8/12/2022,4.89,4.94,5.00,5.00,4.83,4.28,Chicago,within an hour,96,95,Back of the Yards,1.0,f,1 shared bath
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,within an hour,100,97,Clearwater Beach,3.0,f,3 baths
2,5.00,7/21/2023,2.78,12/3/2023,5.00,4.87,5.00,5.00,4.80,4.93,NaN,within an hour,100,100,Lake View East,3.0,f,1 bath
3,4.23,9/2/2023,3.61,12/6/2023,4.00,4.08,4.38,4.08,3.85,4.54,Chicago,within an hour,100,98,Bucktown,1.0,f,1 bath
4,4.85,5/6/2022,3.24,12/6/2023,4.75,4.80,4.83,4.74,4.72,4.74,Chicago,within an hour,100,100,Near South Side,3.0,t,2 baths
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,4.99,1/24/2019,1.32,10/6/2023,4.99,4.98,5.00,4.98,4.96,4.98,Chicago,within a few hours,88,84,Rogers Park,1.0,t,1 private bath
4996,4.90,8/6/2023,13.11,12/11/2023,4.81,4.71,4.75,4.76,4.56,4.98,Chicago,within an hour,100,89,Irving Park,1.0,t,1 bath
4997,5.00,10/3/2019,0.70,11/25/2023,4.97,4.94,4.94,4.97,4.86,4.97,Chicago,within an hour,100,97,North Park,2.0,f,1 bath
4998,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,New York,within an hour,100,97,Cambridge,2.0,f,1 bath


In [103]:
numeric_null_column = df[b.index].select_dtypes(include='number')
numeric_null_column

,review_scores_communication,reviews_per_month,review_scores_cleanliness,review_scores_rating,review_scores_checkin,review_scores_accuracy,review_scores_value,review_scores_location,beds
0,5.00,0.33,4.89,4.94,5.00,5.00,4.83,4.28,1.0
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0
2,5.00,2.78,5.00,4.87,5.00,5.00,4.80,4.93,3.0
3,4.23,3.61,4.00,4.08,4.38,4.08,3.85,4.54,1.0
4,4.85,3.24,4.75,4.80,4.83,4.74,4.72,4.74,3.0
...,...,...,...,...,...,...,...,...,...
4995,4.99,1.32,4.99,4.98,5.00,4.98,4.96,4.98,1.0
4996,4.90,13.11,4.81,4.71,4.75,4.76,4.56,4.98,1.0
4997,5.00,0.70,4.97,4.94,4.94,4.97,4.86,4.97,2.0
4998,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0


In [104]:
numeric_columns_names = numeric_null_column.columns
numeric_columns_names

Index(['review_scores_communication', 'reviews_per_month',
       'review_scores_cleanliness', 'review_scores_rating',
       'review_scores_checkin', 'review_scores_accuracy',
       'review_scores_value', 'review_scores_location', 'beds'],
      dtype='str')

In [105]:
for i in numeric_columns_names:

    df[i] = df[i].fillna(df[i].mean())

    print(df[i].mean())

4.8534259727134925
1.928420919656392
4.74733451237999
4.759537762061127
4.841083606971457
4.793420055569588
4.668380904268754
4.745506441020461
2.289579967689822


In [106]:
cat_null_column = df[b.index].select_dtypes(include='object')
cat_null_column

C:\Users\SART\AppData\Local\Temp\ipykernel_11592\1756435958.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_null_column = df[b.index].select_dtypes(include='object')


,first_review,last_review,host_location,host_response_time,host_response_rate,host_acceptance_rate,host_neighbourhood,host_is_superhost,bathrooms_text
0,6/16/2019,8/12/2022,Chicago,within an hour,96,95,Back of the Yards,f,1 shared bath
1,NaN,NaN,NaN,within an hour,100,97,Clearwater Beach,f,3 baths
2,7/21/2023,12/3/2023,NaN,within an hour,100,100,Lake View East,f,1 bath
3,9/2/2023,12/6/2023,Chicago,within an hour,100,98,Bucktown,f,1 bath
4,5/6/2022,12/6/2023,Chicago,within an hour,100,100,Near South Side,t,2 baths
...,...,...,...,...,...,...,...,...,...
4995,1/24/2019,10/6/2023,Chicago,within a few hours,88,84,Rogers Park,t,1 private bath
4996,8/6/2023,12/11/2023,Chicago,within an hour,100,89,Irving Park,t,1 bath
4997,10/3/2019,11/25/2023,Chicago,within an hour,100,97,North Park,f,1 bath
4998,NaN,NaN,New York,within an hour,100,97,Cambridge,f,1 bath


In [107]:
cat_columns_names = cat_null_column.columns

In [108]:
for i in cat_null_column:

    df[i] = df[i].fillna(df[i].mode()[0])

    print(df[i].mode())

0    6/4/2023
Name: first_review, dtype: str
0    11/26/2023
Name: last_review, dtype: str
0    Chicago
Name: host_location, dtype: object
0    within an hour
Name: host_response_time, dtype: str
0    100
Name: host_response_rate, dtype: str
0    100
Name: host_acceptance_rate, dtype: str
0    Cambridge
Name: host_neighbourhood, dtype: str
0    f
Name: host_is_superhost, dtype: str
0    1 bath
Name: bathrooms_text, dtype: str


In [109]:
df.isnull().sum()

id                                              0
host_id                                         0
host_since                                      0
host_location                                   0
host_response_time                              0
host_response_rate                              0
host_acceptance_rate                            0
host_is_superhost                               0
host_neighbourhood                              0
host_listings_count                             0
host_total_listings_count                       0
host_verifications                              0
host_has_profile_pic                            0
host_identity_verified                          0
neighbourhood_cleansed                          0
latitude                                        0
longitude                                       0
property_type                                   0
room_type                                       0
accommodates                                    0


In [110]:
df['host_verifications']

0                     ['email', 'phone']
1                     ['email', 'phone']
2                              ['phone']
3       ['email', 'phone', 'work_email']
4                     ['email', 'phone']
                      ...               
4995                  ['email', 'phone']
4996                  ['email', 'phone']
4997                  ['email', 'phone']
4998    ['email', 'phone', 'work_email']
4999                  ['email', 'phone']
Name: host_verifications, Length: 5000, dtype: str

In [111]:
for item in ['email', 'phone', 'work_email']:
    df[f'verif_{item}'] = df['host_verifications'].str.contains(f"'{item}'").astype(int)

In [136]:
df['host_response_rate'] = df['host_response_rate'].astype(int)
df['host_acceptance_rate'] = df['host_acceptance_rate'].astype(int)

In [137]:
df.head()

,host_since,host_location,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_neighbourhood,host_listings_count,host_total_listings_count,host_has_profile_pic,host_identity_verified,neighbourhood_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms_text,beds,price,minimum_nights,maximum_nights,minimum_minimum_nights,maximum_minimum_nights,minimum_maximum_nights,maximum_maximum_nights,minimum_nights_avg_ntm,maximum_nights_avg_ntm,has_availability,availability_30,availability_60,availability_90,availability_365,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,first_review,last_review,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month,verif_email,verif_phone,verif_work_email
0,4/9/2018,Chicago,within an hour,96,95,f,Back of the Yards,11,13,t,t,New City,41.794240,-87.661380,Private room in rental unit,Private room,1,1.0,1.0,30.0,32,1125,32,32,1125,1125,32.0,1125.0,t,30,60,90,365,18,0,0,6/16/2019,8/12/2022,4.940000,5.00000,4.890000,5.000000,5.000000,4.280000,4.830000,f,9,1,8,0,0.330000,1,1,0
1,12/23/2014,Chicago,within an hour,100,97,f,Clearwater Beach,1141,2346,t,t,Near North Side,41.892749,-87.620711,Room in hotel,Private room,12,3.0,3.0,1247.0,32,365,32,32,365,365,32.0,365.0,t,25,49,66,135,0,0,0,6/4/2023,11/26/2023,4.759538,4.79342,4.747335,4.841084,4.853426,4.745506,4.668381,t,58,0,58,0,1.928421,1,1,0
2,3/8/2020,Chicago,within an hour,100,100,f,Lake View East,1,1,t,t,Lake View,41.948160,-87.652309,Entire rental unit,Entire home/apt,6,1.0,3.0,214.0,2,45,2,2,45,45,2.0,45.0,t,0,0,0,0,14,14,1,7/21/2023,12/3/2023,4.870000,5.00000,5.000000,5.000000,5.000000,4.930000,4.800000,f,1,1,0,0,2.780000,0,1,0
3,10/22/2015,Chicago,within an hour,100,98,f,Bucktown,55,58,t,t,Hyde Park,41.801030,-87.597160,Entire rental unit,Entire home/apt,2,1.0,1.0,59.0,2,180,2,2,180,180,2.0,180.0,t,23,53,83,355,13,13,3,9/2/2023,12/6/2023,4.080000,4.08000,4.000000,4.380000,4.230000,4.540000,3.850000,t,55,55,0,0,3.610000,1,1,1
4,4/4/2021,Chicago,within an hour,100,100,t,Near South Side,74,75,t,t,Near South Side,41.855770,-87.624130,Entire rental unit,Entire home/apt,6,2.0,3.0,151.0,2,365,2,2,1125,1125,2.0,1125.0,t,0,0,0,0,64,42,3,5/6/2022,12/6/2023,4.800000,4.74000,4.750000,4.830000,4.850000,4.740000,4.720000,t,74,74,0,0,3.240000,1,1,0


In [138]:
df['price'].head()

0      30.0
1    1247.0
2     214.0
3      59.0
4     151.0
Name: price, dtype: float64

In [139]:
df['price'] = df['price'].str.replace(r'[\$,]', '', regex=True).astype(float)

AttributeError: Can only use .str accessor with string values, not floating

In [ ]:
df['price']

0         30.0
1       1247.0
2        214.0
3         59.0
4        151.0
         ...  
4995      91.0
4996      46.0
4997     100.0
4998     169.0
4999      29.0
Name: price, Length: 5000, dtype: float64

In [140]:
clean_text = df['bathrooms_text'].str.replace(r'[^\d.]', '', regex=True)
df['bathrooms_text'] = pd.to_numeric(clean_text, errors='coerce')


AttributeError: Can only use .str accessor with string values, not floating

In [141]:
df['bathrooms_text'].value_counts()

bathrooms_text
1.0     3326
2.0      979
1.5      216
3.0      146
2.5      140
3.5       70
4.0       51
0.0       24
4.5       11
6.0        8
5.5        7
5.0        5
11.0       3
7.0        2
11.5       1
10.0       1
6.5        1
8.0        1
9.5        1
Name: count, dtype: int64

In [142]:
df.head()

,host_since,host_location,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_neighbourhood,host_listings_count,host_total_listings_count,host_has_profile_pic,host_identity_verified,neighbourhood_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms_text,beds,price,minimum_nights,maximum_nights,minimum_minimum_nights,maximum_minimum_nights,minimum_maximum_nights,maximum_maximum_nights,minimum_nights_avg_ntm,maximum_nights_avg_ntm,has_availability,availability_30,availability_60,availability_90,availability_365,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,first_review,last_review,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month,verif_email,verif_phone,verif_work_email
0,4/9/2018,Chicago,within an hour,96,95,f,Back of the Yards,11,13,t,t,New City,41.794240,-87.661380,Private room in rental unit,Private room,1,1.0,1.0,30.0,32,1125,32,32,1125,1125,32.0,1125.0,t,30,60,90,365,18,0,0,6/16/2019,8/12/2022,4.940000,5.00000,4.890000,5.000000,5.000000,4.280000,4.830000,f,9,1,8,0,0.330000,1,1,0
1,12/23/2014,Chicago,within an hour,100,97,f,Clearwater Beach,1141,2346,t,t,Near North Side,41.892749,-87.620711,Room in hotel,Private room,12,3.0,3.0,1247.0,32,365,32,32,365,365,32.0,365.0,t,25,49,66,135,0,0,0,6/4/2023,11/26/2023,4.759538,4.79342,4.747335,4.841084,4.853426,4.745506,4.668381,t,58,0,58,0,1.928421,1,1,0
2,3/8/2020,Chicago,within an hour,100,100,f,Lake View East,1,1,t,t,Lake View,41.948160,-87.652309,Entire rental unit,Entire home/apt,6,1.0,3.0,214.0,2,45,2,2,45,45,2.0,45.0,t,0,0,0,0,14,14,1,7/21/2023,12/3/2023,4.870000,5.00000,5.000000,5.000000,5.000000,4.930000,4.800000,f,1,1,0,0,2.780000,0,1,0
3,10/22/2015,Chicago,within an hour,100,98,f,Bucktown,55,58,t,t,Hyde Park,41.801030,-87.597160,Entire rental unit,Entire home/apt,2,1.0,1.0,59.0,2,180,2,2,180,180,2.0,180.0,t,23,53,83,355,13,13,3,9/2/2023,12/6/2023,4.080000,4.08000,4.000000,4.380000,4.230000,4.540000,3.850000,t,55,55,0,0,3.610000,1,1,1
4,4/4/2021,Chicago,within an hour,100,100,t,Near South Side,74,75,t,t,Near South Side,41.855770,-87.624130,Entire rental unit,Entire home/apt,6,2.0,3.0,151.0,2,365,2,2,1125,1125,2.0,1125.0,t,0,0,0,0,64,42,3,5/6/2022,12/6/2023,4.800000,4.74000,4.750000,4.830000,4.850000,4.740000,4.720000,t,74,74,0,0,3.240000,1,1,0


In [143]:
df.drop(columns=['id', 'host_id', 'host_verifications'], inplace=True)

KeyError: "['id', 'host_id', 'host_verifications'] not found in axis"

In [144]:
df.head()

,host_since,host_location,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_neighbourhood,host_listings_count,host_total_listings_count,host_has_profile_pic,host_identity_verified,neighbourhood_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms_text,beds,price,minimum_nights,maximum_nights,minimum_minimum_nights,maximum_minimum_nights,minimum_maximum_nights,maximum_maximum_nights,minimum_nights_avg_ntm,maximum_nights_avg_ntm,has_availability,availability_30,availability_60,availability_90,availability_365,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,first_review,last_review,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month,verif_email,verif_phone,verif_work_email
0,4/9/2018,Chicago,within an hour,96,95,f,Back of the Yards,11,13,t,t,New City,41.794240,-87.661380,Private room in rental unit,Private room,1,1.0,1.0,30.0,32,1125,32,32,1125,1125,32.0,1125.0,t,30,60,90,365,18,0,0,6/16/2019,8/12/2022,4.940000,5.00000,4.890000,5.000000,5.000000,4.280000,4.830000,f,9,1,8,0,0.330000,1,1,0
1,12/23/2014,Chicago,within an hour,100,97,f,Clearwater Beach,1141,2346,t,t,Near North Side,41.892749,-87.620711,Room in hotel,Private room,12,3.0,3.0,1247.0,32,365,32,32,365,365,32.0,365.0,t,25,49,66,135,0,0,0,6/4/2023,11/26/2023,4.759538,4.79342,4.747335,4.841084,4.853426,4.745506,4.668381,t,58,0,58,0,1.928421,1,1,0
2,3/8/2020,Chicago,within an hour,100,100,f,Lake View East,1,1,t,t,Lake View,41.948160,-87.652309,Entire rental unit,Entire home/apt,6,1.0,3.0,214.0,2,45,2,2,45,45,2.0,45.0,t,0,0,0,0,14,14,1,7/21/2023,12/3/2023,4.870000,5.00000,5.000000,5.000000,5.000000,4.930000,4.800000,f,1,1,0,0,2.780000,0,1,0
3,10/22/2015,Chicago,within an hour,100,98,f,Bucktown,55,58,t,t,Hyde Park,41.801030,-87.597160,Entire rental unit,Entire home/apt,2,1.0,1.0,59.0,2,180,2,2,180,180,2.0,180.0,t,23,53,83,355,13,13,3,9/2/2023,12/6/2023,4.080000,4.08000,4.000000,4.380000,4.230000,4.540000,3.850000,t,55,55,0,0,3.610000,1,1,1
4,4/4/2021,Chicago,within an hour,100,100,t,Near South Side,74,75,t,t,Near South Side,41.855770,-87.624130,Entire rental unit,Entire home/apt,6,2.0,3.0,151.0,2,365,2,2,1125,1125,2.0,1125.0,t,0,0,0,0,64,42,3,5/6/2022,12/6/2023,4.800000,4.74000,4.750000,4.830000,4.850000,4.740000,4.720000,t,74,74,0,0,3.240000,1,1,0


In [145]:
df['host_response_time'].value_counts()

host_response_time
within an hour        4286
within a few hours     433
within a day           230
a few days or more      51
Name: count, dtype: int64

In [146]:
le = LabelEncoder()

In [147]:
cat_columns = df.select_dtypes(include='object')
cat_columns

C:\Users\SART\AppData\Local\Temp\ipykernel_11592\2699812216.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_columns = df.select_dtypes(include='object')


,host_since,host_location,host_response_time,host_is_superhost,host_neighbourhood,host_has_profile_pic,host_identity_verified,neighbourhood_cleansed,property_type,room_type,has_availability,first_review,last_review,instant_bookable
0,4/9/2018,Chicago,within an hour,f,Back of the Yards,t,t,New City,Private room in rental unit,Private room,t,6/16/2019,8/12/2022,f
1,12/23/2014,Chicago,within an hour,f,Clearwater Beach,t,t,Near North Side,Room in hotel,Private room,t,6/4/2023,11/26/2023,t
2,3/8/2020,Chicago,within an hour,f,Lake View East,t,t,Lake View,Entire rental unit,Entire home/apt,t,7/21/2023,12/3/2023,f
3,10/22/2015,Chicago,within an hour,f,Bucktown,t,t,Hyde Park,Entire rental unit,Entire home/apt,t,9/2/2023,12/6/2023,t
4,4/4/2021,Chicago,within an hour,t,Near South Side,t,t,Near South Side,Entire rental unit,Entire home/apt,t,5/6/2022,12/6/2023,t
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,1/6/2019,Chicago,within a few hours,t,Rogers Park,t,t,Rogers Park,Private room in guest suite,Private room,t,1/24/2019,10/6/2023,f
4996,12/8/2022,Chicago,within an hour,t,Irving Park,t,t,Lake View,Entire rental unit,Entire home/apt,t,8/6/2023,12/11/2023,f
4997,7/25/2017,Chicago,within an hour,f,North Park,t,t,North Park,Entire rental unit,Entire home/apt,t,10/3/2019,11/25/2023,f
4998,12/16/2016,New York,within an hour,f,Cambridge,t,t,Lake View,Entire rental unit,Entire home/apt,t,6/4/2023,11/26/2023,f


In [148]:
le = LabelEncoder()

cat_columns = df.select_dtypes(include='object')
cat_columns_names = cat_columns.columns

for i in cat_columns_names:

    df[i] = le.fit_transform(df[i])

C:\Users\SART\AppData\Local\Temp\ipykernel_11592\686184696.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_columns = df.select_dtypes(include='object')


In [149]:
df.head()

,host_since,host_location,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_neighbourhood,host_listings_count,host_total_listings_count,host_has_profile_pic,host_identity_verified,neighbourhood_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms_text,beds,price,minimum_nights,maximum_nights,minimum_minimum_nights,maximum_minimum_nights,minimum_maximum_nights,maximum_maximum_nights,minimum_nights_avg_ntm,maximum_nights_avg_ntm,has_availability,availability_30,availability_60,availability_90,availability_365,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,first_review,last_review,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month,verif_email,verif_phone,verif_work_email
0,1030,23,3,96,95,0,9,11,13,1,1,49,41.794240,-87.661380,28,2,1,1.0,1.0,30.0,32,1125,32,32,1125,1125,32.0,1125.0,1,30,60,90,365,18,0,0,1136,444,4.940000,5.00000,4.890000,5.000000,5.000000,4.280000,4.830000,0,9,1,8,0,0.330000,1,1,0
1,531,23,3,100,97,0,35,1141,2346,1,1,46,41.892749,-87.620711,36,2,12,3.0,3.0,1247.0,32,365,32,32,365,365,32.0,365.0,1,25,49,66,135,0,0,0,1242,152,4.759538,4.79342,4.747335,4.841084,4.853426,4.745506,4.668381,1,58,0,58,0,1.928421,1,1,0
2,873,23,3,100,100,0,87,1,1,1,1,37,41.948160,-87.652309,10,0,6,1.0,3.0,214.0,2,45,2,2,45,45,2.0,45.0,1,0,0,0,0,14,14,1,1332,201,4.870000,5.00000,5.000000,5.000000,5.000000,4.930000,4.800000,0,1,1,0,0,2.780000,0,1,0
3,229,23,3,100,98,0,22,55,58,1,1,33,41.801030,-87.597160,10,0,2,1.0,1.0,59.0,2,180,2,2,180,180,2.0,180.0,1,23,53,83,355,13,13,3,1674,209,4.080000,4.08000,4.000000,4.380000,4.230000,4.540000,3.850000,1,55,55,0,0,3.610000,1,1,1
4,1004,23,3,100,100,1,111,74,75,1,1,47,41.855770,-87.624130,10,0,6,2.0,3.0,151.0,2,365,2,2,1125,1125,2.0,1125.0,1,0,0,0,0,64,42,3,1073,209,4.800000,4.74000,4.750000,4.830000,4.850000,4.740000,4.720000,1,74,74,0,0,3.240000,1,1,0


In [151]:
sc = StandardScaler()


In [152]:
sc.fit_transform(df)

array([[ 0.36895661, -0.39322028,  0.37060792, ...,  0.2781259 ,
         0.020004  , -0.53386956],
       [-0.60824225, -0.39322028,  0.37060792, ...,  0.2781259 ,
         0.020004  , -0.53386956],
       [ 0.06150125, -0.39322028,  0.37060792, ..., -3.59549393,
         0.020004  , -0.53386956],
       ...,
       [ 1.2012402 , -0.39322028,  0.37060792, ...,  0.2781259 ,
         0.020004  , -0.53386956],
       [-0.68069988,  2.04466688,  0.37060792, ...,  0.2781259 ,
         0.020004  ,  1.87311673],
       [ 0.71949487, -0.39322028, -1.40094045, ...,  0.2781259 ,
         0.020004  , -0.53386956]], shape=(5000, 54))

In [166]:
df.dropna(inplace=True)

In [167]:
X = df.drop(columns='price')
y = df['price']

In [168]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [169]:
linear_model = LinearRegression()

In [171]:
linear_model.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](53,)","[ 0.02, 0. ,-0. ,..., 0. ,-0. , 0. ]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](53,)","['host_since','host_location','host_response_time',...,'verif_email', 'verif_phone','verif_work_email']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,113.8
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,53
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(9)


In [173]:
pred = linear_model.predict(X_test)

In [175]:
MSE_LINEAR = mean_squared_error(pred, y_test)
MSE_LINEAR

69786.59473217312

In [182]:
desc_tree = DecisionTreeRegressor()
desc_tree.fit(X_train, y_train)
pred = desc_tree.predict(X_test)
MSE_TREE = mean_squared_error(pred, y_test)
MSE_TREE

64264.11882870525

In [193]:
desc_tree = DecisionTreeRegressor(max_depth=5000, random_state=42, min_samples_leaf=5000)
desc_tree.fit(X_train, y_train)
pred = desc_tree.predict(X_test)
MSE_TREE = mean_squared_error(pred, y_test)
MSE_TREE

63144.94262812959

In [199]:
import pickle

# Save your model (replace 'best_model' with your model variable name)
with open('airbnb_model.pkl', 'wb') as file:
    pickle.dump(best_model, file)

print("Model saved successfully!")


Model saved successfully!


In [202]:
# Sort descending so the most important features appear at the top
importance_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': desc_tree.feature_importances_
}).sort_values(by='Importance', ascending=False)

importance_df.head(10)


,Feature,Importance
0,host_since,0.0
1,host_location,0.0
2,host_response_time,0.0
3,host_response_rate,0.0
4,host_acceptance_rate,0.0
5,host_is_superhost,0.0
6,host_neighbourhood,0.0
7,host_listings_count,0.0
8,host_total_listings_count,0.0
9,host_has_profile_pic,0.0


In [187]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error

# 1. Base model
dt = DecisionTreeRegressor(random_state=42)

# 2. Define the parameter grid to search
param_grid = {
    'max_depth': [5, 10, 15, 20, None],
    'min_samples_split': [2, 10, 20],
    'min_samples_leaf': [1, 5, 10, 20],
    'criterion': ['squared_error', 'friedman_mse']
}

# 3. Setup GridSearchCV (cv=5 means 5-fold cross validation)
grid_search = GridSearchCV(
    estimator=dt,
    param_grid=param_grid,
    scoring='neg_mean_squared_error',  # sklearn uses negative MSE for optimization
    cv=5,
    n_jobs=-1,  # Uses all CPU cores to run faster
    verbose=1
)

# 4. Fit the grid search on training data
grid_search.fit(X_train, y_train)

# 5. Check the best parameters and score
print("Best Parameters:", grid_search.best_params_)
print("Best CV MSE:", -grid_search.best_score_)

# 6. Predict on test set using the best model
best_model = grid_search.best_estimator_
pred = best_model.predict(X_test)

# 7. Evaluate test MSE
test_mse = mean_squared_error(y_test, pred)
print("Test MSE:", test_mse)


Fitting 5 folds for each of 120 candidates, totalling 600 fits
Best Parameters: {'criterion': 'squared_error', 'max_depth': 5, 'min_samples_leaf': 1, 'min_samples_split': 10}
Best CV MSE: 2522047.228700987
Test MSE: 10040514.930885972


c:\Users\SART\Desktop\Practise\venv\Lib\site-packages\sklearn\model_selection\_validation.py:493: FitFailedWarning: 
300 fits failed out of a total of 600.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
43 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\SART\Desktop\Practise\venv\Lib\site-packages\sklearn\model_selection\_validation.py", line 855, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\SART\Desktop\Practise\venv\Lib\site-packages\sklearn\base.py", line 1393, in wrapper
    estimator._validate_params()
  File "c:\Users\SART\Desktop\Practise\venv\Lib\site-packages\sklearn\base.py", line 554, in _validate_params
    validate_parameter_constraints(
  Fil

In [188]:
# 5. Check the best parameters and score
print("Best Parameters:", grid_search.best_params_)
print("Best CV MSE:", -grid_search.best_score_)

Best Parameters: {'criterion': 'squared_error', 'max_depth': 5, 'min_samples_leaf': 1, 'min_samples_split': 10}
Best CV MSE: 2522047.228700987


In [190]:
random_forest = RandomForestRegressor()
random_forest.fit(X_train, y_train)
pred = random_forest.predict(X_test)
rand_forst = mean_squared_error(pred, y_test)
rand_forst

238007.7551704313

In [197]:
desc_tree = DecisionTreeRegressor(max_depth=2000, random_state=42, min_samples_leaf=2000)
desc_tree.fit(X_train, y_train)
pred = desc_tree.predict(X_test)
MSE_TREE = mean_squared_error(pred, y_test)
MSE_TREE

63144.94262812959

In [203]:
desc_tree.feature_importances_

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0.])

In [208]:
from sklearn.linear_model import LinearRegression
import pandas as pd

# 1. Train Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)

# 2. Get the weights (coefficients) for EVERY feature
weights_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Weight': lr.coef_
}).sort_values(by='Weight', ascending=False)

# 3. View the top positive weights
print("Top Positive Weights (Increase Price):")
weights_df.head(10)


Top Positive Weights (Increase Price):


,Feature,Weight
36,last_review,0.543450
8,host_total_listings_count,0.065018
35,first_review,0.021380
21,minimum_minimum_nights,0.018549
19,minimum_nights,0.018423
0,host_since,0.016257
25,minimum_nights_avg_ntm,0.009474
22,maximum_minimum_nights,0.009038
26,maximum_nights_avg_ntm,0.004008
6,host_neighbourhood,0.002486


In [205]:
importance_df[importance_df['Importance'] > 0]


,Feature,Importance
